# Simon — Personal Tracker Analysis

Time use, sleep, mood, and productivity. Pull → explore → find better patterns.

## 0. Setup

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np

from personal_tracker.analysis import TrackerDB

plt.rcParams['figure.figsize'] = (10, 4)
plt.rcParams['axes.spines.top'] = False
plt.rcParams['axes.spines.right'] = False

In [ ]:
# Pull fresh data from Pi
# At home:  db.sync('192.168.88.9')
# Away:     db.sync('100.93.132.118')

db = TrackerDB()
db.sync('192.168.88.9')

## 1. Today's Summary

In [ ]:
db.daily_summary()

In [ ]:
# Specific date
# db.daily_summary('2026-03-25')

## 2. Time Use — Where Does the Day Go?

In [ ]:
blocks = db.time_blocks()
print(f"{len(blocks)} time blocks logged")
blocks.head()

In [ ]:
# Overall time allocation across all days
totals = db.time_use_totals()

if not totals.empty:
    totals['hours'] = totals['duration_min'] / 60
    totals['pct']   = (totals['duration_min'] / totals['duration_min'].sum() * 100).round(1)

    fig, axes = plt.subplots(1, 2, figsize=(13, 5))

    axes[0].barh(totals['category'], totals['hours'], color='#1a7a4a')
    axes[0].set_title('Total Hours by Category')
    axes[0].set_xlabel('Hours')
    axes[0].invert_yaxis()

    axes[1].pie(totals['duration_min'], labels=totals['category'],
                autopct='%1.0f%%', startangle=90,
                colors=plt.cm.Set3.colors[:len(totals)])
    axes[1].set_title('Time Share')

    plt.tight_layout()
    plt.show()
    display(totals[['category','hours','pct']].rename(columns={'hours':'Hours','pct':'%'}))
else:
    print('No time blocks logged yet.')

In [ ]:
# Daily time use — stacked bar by category
by_day = db.time_use_by_day()

if not by_day.empty:
    pivoted = by_day.pivot(index='date', columns='category', values='duration_min').fillna(0)
    pivoted_hr = pivoted / 60

    pivoted_hr.plot(kind='bar', stacked=True, figsize=(12, 5),
                    colormap='Set3', width=0.7)
    plt.title('Daily Time Use (hours)')
    plt.xlabel('')
    plt.ylabel('Hours')
    plt.xticks(rotation=45)
    plt.legend(bbox_to_anchor=(1.01, 1), loc='upper left', fontsize=8)
    plt.tight_layout()
    plt.show()
else:
    print('Need at least one completed time block.')

In [ ]:
# Productive hours per day (job search + LinkedIn + coding + learning)
prod = db.productive_hours_per_day()

if not prod.empty:
    plt.bar(prod['date'], prod['productive_hours'], color='#1a7a4a')
    plt.axhline(2, color='orange', linestyle='--', alpha=0.7, label='2h target')
    plt.axhline(4, color='red',    linestyle='--', alpha=0.7, label='4h stretch')
    plt.title('Productive Hours per Day')
    plt.ylabel('Hours')
    plt.xticks(rotation=45)
    plt.legend()
    plt.tight_layout()
    plt.show()

    print(f"Average productive hours/day: {prod['productive_hours'].mean():.1f}h")
    print(f"Best day: {prod.loc[prod['productive_hours'].idxmax(), 'date']} ({prod['productive_hours'].max():.1f}h)")
else:
    print('No productive time blocks yet.')

In [ ]:
# What hour of day are you most productive?
if not blocks.empty:
    from personal_tracker.analysis import PRODUCTIVE_CATEGORIES
    prod_blocks = blocks[blocks['category'].isin(PRODUCTIVE_CATEGORIES)].dropna(subset=['duration_min'])

    if not prod_blocks.empty:
        by_hour = prod_blocks.groupby('hour')['duration_min'].sum() / 60
        plt.bar(by_hour.index, by_hour.values, color='#1a7a4a')
        plt.title('Productive Time by Hour of Day')
        plt.xlabel('Hour')
        plt.ylabel('Total Hours')
        plt.xticks(range(6, 23))
        plt.tight_layout()
        plt.show()
    else:
        print('No productive blocks yet.')

## 3. Sleep

In [ ]:
sleep = db.sleep()
sleep.head()

In [ ]:
s = sleep.dropna(subset=['night_sleep_hr'])
s = s[s['night_sleep_hr'] > 0]

if not s.empty:
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))

    axes[0].bar(s['date'], s['night_sleep_hr'], color='#2c3e6b')
    axes[0].axhline(7, color='orange', linestyle='--', alpha=0.7, label='7h')
    axes[0].axhline(8, color='green',  linestyle='--', alpha=0.7, label='8h')
    axes[0].set_title('Night Sleep (hours)')
    axes[0].tick_params(axis='x', rotation=45)
    axes[0].legend()

    sq = s.dropna(subset=['sleep_quality'])
    if not sq.empty:
        axes[1].plot(sq['date'], sq['sleep_quality'], 'o-', color='#7b5ea7')
        axes[1].set_ylim(0, 5.5)
        axes[1].set_title('Sleep Quality (1-5)')
        axes[1].tick_params(axis='x', rotation=45)
    else:
        axes[1].text(0.5, 0.5, 'Log sleep quality\non wake-up', ha='center', va='center', transform=axes[1].transAxes)

    plt.tight_layout()
    plt.show()
else:
    print('Need 2+ days of wake/bed data.')

In [ ]:
# Does sleep quality predict productive hours?
prod = db.productive_hours_per_day()
sq   = sleep.dropna(subset=['sleep_quality'])[['date', 'sleep_quality']]
sq['date'] = sq['date'].astype(str)

if not prod.empty and not sq.empty:
    merged = prod.merge(sq, on='date', how='inner')
    if not merged.empty:
        plt.scatter(merged['sleep_quality'], merged['productive_hours'],
                    s=100, color='#1a7a4a', alpha=0.7)
        plt.xlabel('Sleep Quality (1-5)')
        plt.ylabel('Productive Hours')
        plt.title('Sleep Quality vs Productive Hours')
        plt.xticks([1,2,3,4,5])
        plt.tight_layout()
        plt.show()
    else:
        print('Need more overlapping data.')

## 4. Mood & Energy

In [ ]:
mood = db.mood()
print(f"{len(mood)} mood logs")
mood.head()

In [ ]:
if not mood.empty:
    daily_mood = mood.groupby('date').agg(
        avg_energy=('energy', 'mean'),
        avg_mood=('mood', 'mean')
    ).reset_index()

    fig, axes = plt.subplots(1, 2, figsize=(12, 4))

    axes[0].plot(daily_mood['date'], daily_mood['avg_energy'], 'o-', color='#e07c2a', label='Energy')
    axes[0].plot(daily_mood['date'], daily_mood['avg_mood'],   'o-', color='#7b5ea7', label='Mood')
    axes[0].set_ylim(0, 5.5)
    axes[0].set_title('Avg Daily Mood & Energy')
    axes[0].tick_params(axis='x', rotation=45)
    axes[0].legend()

    axes[1].scatter(daily_mood['avg_energy'], daily_mood['avg_mood'],
                    s=100, color='#1a7a4a', alpha=0.7)
    axes[1].set_xlabel('Avg Energy')
    axes[1].set_ylabel('Avg Mood')
    axes[1].set_title('Energy vs Mood')
    axes[1].set_xlim(0, 5.5)
    axes[1].set_ylim(0, 5.5)

    plt.tight_layout()
    plt.show()
else:
    print('No mood logged yet.')

In [ ]:
# Does energy predict productive hours?
if not mood.empty and not prod.empty:
    daily_energy = mood.groupby('date')['energy'].mean().reset_index(name='avg_energy')
    merged = prod.merge(daily_energy, on='date', how='inner')
    if not merged.empty:
        plt.scatter(merged['avg_energy'], merged['productive_hours'],
                    s=100, color='#e07c2a', alpha=0.7)
        plt.xlabel('Avg Energy (1-5)')
        plt.ylabel('Productive Hours')
        plt.title('Energy vs Productive Hours')
        plt.tight_layout()
        plt.show()

## 5. Food

In [ ]:
food = db.food()
print(f"{len(food)} food events")
food.head()

In [ ]:
if not food.empty:
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))

    food['type'].value_counts().plot.bar(ax=axes[0], color=['#5ab25a','#f5c842','#4a90d9'])
    axes[0].set_title('Meal / Snack / Drink Split')
    axes[0].tick_params(axis='x', rotation=0)

    axes[1].hist(food['hour'], bins=range(6, 23), color='#5ab25a', rwidth=0.8)
    axes[1].set_title('Food Events by Hour')
    axes[1].set_xlabel('Hour')

    plt.tight_layout()
    plt.show()
else:
    print('No food logged yet.')

## 6. Exercise

In [ ]:
ex = db.exercise()
print(f"{len(ex)} exercise sessions")
ex.head()

In [ ]:
if not ex.empty:
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))

    ex['type'].value_counts().plot.barh(ax=axes[0], color='#e07c2a')
    axes[0].set_title('Exercise Types')
    axes[0].invert_yaxis()

    ex_by_date = ex.groupby('date')['duration_min'].sum().reset_index()
    axes[1].bar(ex_by_date['date'], ex_by_date['duration_min'], color='#e07c2a')
    axes[1].set_title('Exercise Minutes per Day')
    axes[1].tick_params(axis='x', rotation=45)

    plt.tight_layout()
    plt.show()

    print(f"Sessions: {len(ex)}")
    print(f"Total time: {ex['duration_min'].sum():.0f} min ({ex['duration_min'].sum()/60:.1f}h)")
    print(f"Avg duration: {ex['duration_min'].mean():.0f} min")
else:
    print('No exercise logged yet. Go touch grass.')

## 7. Scratch — Your Analysis Here

In [ ]:
# All DataFrames: sleep, blocks, mood, food, ex
# db.time_use_by_day()  — pivot-ready daily category totals
# db.productive_hours_per_day()  — job search focus hours

